# PR #242 review — runnable checks

Review of [PR #242](https://github.com/Differentiable-Electron-Crystallography/diffBloch/pull/242),
*Remove Klar window from convergence test beam selection* (branch `convergence-test-sg-cut`).

The PR replaces the convergence test's seed beam selection: instead of pruning with the
Klar window (`select_beams`), it keeps beams with `|Sg| <= sg_max` (`select_beams_sg`).
**Removing Klar from beam selection is right** — Klar is a scoring rule, and AGENTS.md says
the SOLVE and SCORED sets stay independent. The question this notebook answers is whether
the *replacement* rule is right.

This notebook checks the two findings that are measurable. The full review is at
<https://gist.github.com/imaitland/c9a2eb9723f87e1059cee814088a2114>.

**Run it against the PR branch**, since `select_beams_sg` only exists there:

```bash
git checkout convergence-test-sg-cut
uv run jupyter lab reviews/pr-242-convergence-seed.ipynb
```

Cells carry no committed output (house rule). The numbers measured on one machine are
recorded inline in markdown so the evidence reads without executing.

## Setup

Read-only: `load_config` rather than `load_experiment`, so no `experiment.lock` is written
or verified. `setup_datasets` gives the candidate `Plan` the convergence test seeds from —
`convergence_test_experiment` takes `dataset.plan.orientations[:n_orientations]` (default 1)
and applies the seed rule to it.

In [ ]:
from dataclasses import replace
from pathlib import Path

import numpy as np

from diffBloch.config import load_config
from diffBloch.core.crystal import orientation_basis
from diffBloch.core.dynamical import excitation_errors
from diffBloch.core.reciprocal import g_vectors
from diffBloch.io import read_experimental_data, read_structure
from diffBloch.preprocess import select_beams, select_beams_sg, setup_datasets
from diffBloch.preprocess.orientation import rocking_curve_tilts

REPO = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "examples/Colmey_et_al_2026/data").is_dir()
)  # so the notebook runs from reviews/ or from the repo root alike
DATA = REPO / "examples/Colmey_et_al_2026/data"
EXAMPLES = ["quartz-no-abs", "cspbbr3-no-abs", "borane-no-abs"]  # the three the PR benchmarks


def load_seed(root: Path):
    '''The first-rotation candidate Plan the convergence test seeds from, plus its config.'''
    cfg = load_config(root / "experiment.yaml")
    refs = cfg.inputs.exp_data
    refs = tuple(refs) if isinstance(refs, list) else (refs,)
    structure = read_structure(
        root / cfg.inputs.structure, load_hydrogens=cfg.inputs.load_hydrogens
    )
    records = tuple(read_experimental_data(root / ref) for ref in refs)
    _refinement, datasets = setup_datasets(structure, records, cfg)
    dataset = datasets[0]
    seed = replace(dataset.plan, orientations=dataset.plan.orientations[:1])
    return cfg, dataset, seed


def beam_hkls(plan) -> set[tuple[int, int, int]]:
    return {tuple(int(v) for v in row) for row in np.asarray(plan.orientations[0].beam_hkl)}

## Finding 2 — the new rule is *different*, not wider

The PR's description says the old seed "was missing significant beams". But Klar's threshold
is g-dependent (`rsg * |g_lever| * theta_semi`) while `|Sg| <= sg_max` is a flat constant, so
swapping them moves beams **both ways**:

- it *admits* low-lever reflections near the goniometer axis (small `|g_lever|`, which Klar
  always dropped — they never sweep through the Ewald sphere), and
- it *drops* high-lever reflections with `sg_max < |Sg| < rsg * |g_lever| * theta_semi`.

The PR reports only net "fixed HKL" counts, which hides the second effect.

In [ ]:
rows = []
for name in EXAMPLES:
    cfg, dataset, seed = load_seed(DATA / name)
    policy = cfg.blochwave.to_policy()
    klar = beam_hkls(select_beams(cfg.blochwave.to_beam_selection(dataset.integration))(seed))
    sg_cut = beam_hkls(select_beams_sg(policy.sg_max)(seed))
    rows.append(
        {
            "example": name,
            "pool": len(beam_hkls(seed)),
            "klar (main)": len(klar),
            "sg_max (PR)": len(sg_cut),
            "both": len(klar & sg_cut),
            "dropped by PR": len(klar - sg_cut),
            "added by PR": len(sg_cut - klar),
        }
    )

header = list(rows[0])
widths = [max(len(h), *(len(str(r[h])) for r in rows)) for h in header]
print("  ".join(h.rjust(w) for h, w in zip(header, widths)))
for r in rows:
    print("  ".join(str(r[h]).rjust(w) for h, w in zip(header, widths)))

Measured (one machine, PR branch `convergence-test-sg-cut`):

| example | pool | klar (main) | sg_max (PR) | both | dropped by PR | added by PR |
|---|---|---|---|---|---|---|
| quartz-no-abs  |  3801 |  41 |  29 |  20 |  21 |   9 |
| cspbbr3-no-abs | 40617 | 456 | 519 | 381 |  75 | 138 |
| borane-no-abs  | 39801 | 423 | 364 | 293 | 130 |  71 |

The two rules barely agree. Borane keeps 293 of 423 Klar beams and throws away 130 — the
net "-59" the fixed-HKL count would suggest is the difference of two large, opposite
movements. **Finding 2 holds:** this is a different bias, not a wider seed, and the PR
description should say so.

## Finding 1 — `sg_max` is cut at the central tilt only

`_reselect_sg` evaluates `excitation_errors` at `cp.orientation`, the **untilted nominal
orientation**. Everywhere else `sg_max` means `|Sg| < sg_max` *at a segment's boundary
tilts* (`couple_beams`), i.e. across the sweep.

So the test: for the beams the new rule drops, does `|Sg|` fall under `sg_max` anywhere in
the rocking sweep? If it does, those beams are genuinely excited during the rotation —
`couple_beams` will select them into the SOLVE set — and they are excluded from the
convergence comparison set only because the cut was taken at tilt 0.

In [ ]:
def sweep_sg(seed, cfg, dataset, hkl: np.ndarray) -> np.ndarray:
    '''|Sg| for each hkl at every rocking-curve tilt: (n_tilts, n_hkl).'''
    cp = seed.orientations[0]
    rocking = cfg.blochwave.to_rocking_curve(dataset.integration)
    tilts = rocking_curve_tilts(
        rocking.integration.semiangle, rocking.sampling, geometry=rocking.integration.geometry
    )
    nominal = orientation_basis(
        np.asarray(seed.structure_factor_grid.cell), np.asarray(cp.orientation)
    )
    return np.stack(
        [
            np.abs(excitation_errors(g_vectors(hkl, nominal @ tilt.T), cp.energy, u0=cp.u0))
            for tilt in tilts
        ]
    )


summary = {}
for name in EXAMPLES:
    cfg, dataset, seed = load_seed(DATA / name)
    policy = cfg.blochwave.to_policy()
    klar = beam_hkls(select_beams(cfg.blochwave.to_beam_selection(dataset.integration))(seed))
    sg_cut = beam_hkls(select_beams_sg(policy.sg_max)(seed))

    dropped = np.array(sorted(klar - sg_cut), dtype=np.int64)
    sg = sweep_sg(seed, cfg, dataset, dropped)
    min_over_sweep = sg.min(axis=0)
    at_centre = sg[len(sg) // 2]
    excited = min_over_sweep <= policy.sg_max
    summary[name] = (min_over_sweep, policy.sg_max)

    print(f"{name}: {dropped.shape[0]} beams dropped by the sg-cut")
    print(f"  reach |Sg| <= sg_max in the sweep : {excited.sum()} / {excited.size}"
          f" ({100 * excited.mean():.0f}%)")
    print(f"  median |Sg| at the central tilt   : {np.median(at_centre):.5f}")
    print(f"  median min |Sg| over the sweep    : {np.median(min_over_sweep):.5f}"
          f"   (sg_max = {policy.sg_max})")

Measured:

| example | sg_max | dropped | excited somewhere in sweep | median \|Sg\| at centre | median min \|Sg\| over sweep |
|---|---|---|---|---|---|
| quartz-no-abs  | 0.01 |  21 | 21 (**100%**)  | 0.01668 | 0.00025 |
| cspbbr3-no-abs | 0.02 |  75 | 75 (**100%**)  | 0.02451 | 0.00039 |
| borane-no-abs  | 0.01 | 130 | 130 (**100%**) | 0.01557 | 0.00014 |

**Every single beam** the new rule drops, in all three examples, becomes excited during the
rocking sweep. These are not marginal cases: quartz's median minimum `|Sg|` over the sweep
is `0.00025`, forty times *below* the cutoff, against `0.01668` at the central tilt; borane's
is `0.00014`, seventy times below. They are not weakly-excited beams; they are beams measured
away from tilt 0.

This matters because the seed's `beam_hkl` becomes `op.alignment.hkl` → `comparison_hkl` in
`run_convergence`, the fixed scored domain of the convergence R-factor. The test now
measures convergence over a sample biased toward centrally-excited reflections — the ones
*least* sensitive to tilt sampling, which is exactly what the `tilt_steps` sweep is meant to
resolve. Borane's settled `tilt_steps` moving 110 → 125 is consistent with that.

**Suggested fix:** take the cut as `min |Sg|` over the rocking tilts rather than at tilt 0,
matching what `couple_beams` means by `sg_max`.

In [ ]:
# Where the dropped beams sit relative to the cutoff.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(summary), figsize=(4 * len(summary), 3.2), constrained_layout=True)
for ax, (name, (min_sg, sg_max)) in zip(np.atleast_1d(axes), summary.items()):
    ax.hist(min_sg, bins=30, color="#4C78A8")
    ax.axvline(sg_max, color="#E45756", ls="--", label=f"sg_max = {sg_max}")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("min |Sg| over the rocking sweep")
    ax.set_ylabel("beams dropped by the sg-cut")
    ax.legend(fontsize=8)
fig.suptitle("Beams the PR's seed drops are excited during the sweep", fontsize=11)
plt.show()

## Cheap static checks (findings 4, 5, 7, 10)

These need no simulation — they are facts about the tree. Run from the repo root on the PR
branch.

In [ ]:
import subprocess


def sh(*args: str) -> str:
    out = subprocess.run(args, cwd=REPO, capture_output=True, text=True)
    return out.stdout.strip()


print("4. tests mentioning select_beams_sg:")
print("  ", sh("grep", "-rl", "select_beams_sg", "tests/") or "(none)")

print("5. CLI subcommand vs docs:")
print("  ", " ".join(sh("grep", "-o", 'args.command == "[a-z_-]*"',
                        "src/diffBloch/app/cli.py").split()[2::3]))
print("  ", sh("grep", "-rh", "diffbloch convergence", "docs/workflow.md",
               "docs/convergence-testing.md"))

print("7. beams.py __all__:")
print("  ", sh("grep", "-n", "^__all__", "src/diffBloch/preprocess/steps/beams.py"))

print("10. _pool names in steps/README.md vs definitions in src:")
print("   README:", sh("grep", "-o", "[a-z_]*_pool", "src/diffBloch/preprocess/steps/README.md"))
print("   defined:", sh("grep", "-rho", "def [a-z_]*_pool", "src/diffBloch/") or "(none)")

Expected on the PR branch:

- **4.** no test file mentions `select_beams_sg` — a new public `Plan -> Plan` step with a
  selection predicate and a 000-retention branch ships with no test.
- **5.** the subcommand is `convergence_test` (underscore), the only other multi-word
  subcommand is `lock-experiment` (hyphen), and both `docs/workflow.md` and
  `docs/convergence-testing.md` already document `diffbloch convergence-test`. The hyphen
  matches convention *and* fixes two docs pages that are currently wrong; the underscore
  keeps them wrong.
- **7.** `beams.py.__all__` is `["build_orientation_plans", "klar_beam_mask", "select_beams"]`
  — `select_beams_sg` is missing, though it is in `preprocess/__init__.py.__all__`. Its
  sibling is in both.
- **10.** `steps/README.md` names `convergence_test_pool` and `cover_pool`; neither exists
  (nor did `converge_pool` on main). Pre-existing rot the rename propagated faithfully.

## Not checked here

Findings 3, 6, 8 and 9 are judgement or prose, not measurements — read them in the
[gist](https://gist.github.com/imaitland/c9a2eb9723f87e1059cee814088a2114). I also did not
reproduce the PR's wall-clock table, and this notebook only ever looks at the **first**
rotation of each dataset, which is what `convergence_test_experiment` defaults to
(`n_orientations=1`).